In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *
import os
import sys
project_path=os.path.join(os.getcwd(),'..','..','..')
sys.path.append(project_path)

os.listdir(project_path)
from spotify_dab.utils.transformation import reusable

###DimUser

In [0]:
df = spark.read.format("parquet") \
    .load("abfss://bronze@storageazureprojectfirst.dfs.core.windows.net/DimUser/")
    

In [0]:
df.where("user_id = 53").display()

###AutoLoader

###Read Bronze with Auto Loader

In [0]:
df_user=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimUser/checkpoint")\
    .load("abfss://bronze@storageazureprojectfirst.dfs.core.windows.net/DimUser")

In [0]:

df_user_obj = reusable()

df_user = df_user_obj.dropColumn(df_user, ['_rescued_data'])


###Write to Silver

In [0]:
query = (
    df_user.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimUser/checkpoint/"
    )
    .trigger(availableNow=True)
    .option("path","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimUser/data/")
    .toTable("spotify_catalog.silver.DimUser")
)
query.awaitTermination()

In [0]:
df_preview = spark.read.format("delta").load(
    "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimUser/data/"
)
display(df_preview)

###DimArtist

In [0]:
df_art=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimArtist/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureprojectfirst.dfs.core.windows.net/DimArtist")

In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumn(
    df_art,
    ["_rescued_data"]
)


In [0]:
query = (
    df_art.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimArtist/checkpoint"
    )
    .trigger(availableNow=True)
    .option("path","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimArtist/data")
    .toTable('spotify_catalog.silver.DimArtist')
)

query.awaitTermination()

In [0]:
df_artist_silver = spark.read.format("delta").load(
    "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimArtist/data/"
)

display(df_artist_silver)

###DimTrack

In [0]:
df_track=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimTrack/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureprojectfirst.dfs.core.windows.net/DimTrack")

In [0]:
df_track_obj = reusable()

df_track = df_track_obj.dropColumn(
    df_track,
    ["_rescued_data"]
)



In [0]:
df_track=df_track.withColumn("durationFlag",when(col("duration_sec")<150,"low")\
                                            .when((col("duration_sec")>=150) & (col("duration_sec")<300),"medium")\
                                            .when(col("duration_sec")>=300,"high")
                                            )

In [0]:
df_track=df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))

In [0]:
query = (
    df_track.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimTrack/checkpoint"
    )
    .trigger(availableNow=True)
    .option("path","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimTrack/data")
    .toTable('spotify_catalog.silver.DimTrack')
)

query.awaitTermination()

In [0]:
df_track_silver = spark.read.format("delta").load(
    "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimTrack/data/"
)

display(df_track_silver)

###DimDate

In [0]:
df_date=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimDate/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureprojectfirst.dfs.core.windows.net/DimDate")

In [0]:
df_date_obj = reusable()

df_date = df_date_obj.dropColumn(
    df_date,
    ["_rescued_data"]
)


In [0]:
query = (
    df_date.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimDate/checkpoint"
    )
    .trigger(availableNow=True)
    .option("path","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimDate/data")
    .toTable('spotify_catalog.silver.DimDate')
)

query.awaitTermination()

In [0]:
df_date_silver = spark.read.format("delta").load(
    "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/DimDate/data/"
)

display(df_date_silver)

###FactStream

In [0]:
df_stream=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
    .option("cloudFiles.schemaLocation","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/FactStream/checkpoint")\
    .option("cloudFiles.schemaEvolutionMode","addNewColumns")\
    .load("abfss://bronze@storageazureprojectfirst.dfs.core.windows.net/FactStream")

In [0]:
df_stream_obj = reusable()

df_stream = df_stream_obj.dropColumn(
    df_stream,
    ["_rescued_data"]
)


In [0]:
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/FactStream/checkpoint"
    )
    .trigger(availableNow=True)
    .option("path","abfss://silver@storageazureprojectfirst.dfs.core.windows.net/FactStream/data")
    .toTable('spotify_catalog.silver.FactStream')
)

query.awaitTermination()

In [0]:
df_stream_silver = spark.read.format("delta").load(
    "abfss://silver@storageazureprojectfirst.dfs.core.windows.net/FactStream/data/"
)

display(df_stream_silver)

In [0]:
%sql
use catalog spotify_catalog;
select * from spotify_catalog.silver.dimtrack;

In [0]:
%sql
select * from silver.dimuser where user_id=53